# 490 — Pooling results (read me)

The **results page** for `04_FBM_Pooling`. It computes nothing heavy — it reads the artifacts
`410`/`420`/`430` wrote and lays them out with the story and a guide to reading each figure. Run
`410 → 420 → 430` first, then run this top-to-bottom.

---

## What was asked
A **confirmatory** test, built on the same per-electrode ERSPs as clustering:

1. **Do contacts respond in a-priori time zones?** Pool ERSP power inside hand-defined windows —
   **perception**, **pre_articulation**, **audio** — and qualify each contact with a
   *window-restricted* version of clustering's high-activity gate.
2. **Where do the responsive contacts cluster?** Map qualifiers to Yeo-7/17 networks and
   Desikan-Killiany gyri, and summarize their anatomical purity + spatial compactness.

## Built to be robust — two window shapes
- **Boxcar** (primary): equal weight across the zone — the straight hypothesis test.
- **Gaussian** (robustness): centre-weighted, so it down-weights the edges and tolerates
  latency jitter. **If boxcar ≈ Gaussian** (qualifier counts, anatomy purity), the zone is real
  and not an artefact of exactly where you drew the window. **If they diverge**, the effect sits
  near a window edge — treat it with caution.

## Two feature sets
- **`hg`** — 70–150 Hz high-gamma line, the canonical task-response marker.
- **`bands15`** — each of the 15 bands separately, so band-specific zone effects are visible.

## How to read each output
| Output | What it is | How to read it |
|---|---|---|
| **Qualifier summary** | distinct qualifying contacts per condition × zone × shape × sign | the headline: which zones recruit many contacts, and whether `+`/`−` dominate |
| **Per-zone anatomy** | purity, entropy, top-3 regions per zone | high purity / low entropy = the zone's contacts share an anatomy |
| **Spatial compactness** | mm spread of each zone's contacts (hemisphere-mirrored) | small `distance_mm` = a tight anatomical cluster |
| **Surface renders** | contacts on fsaverage, red(+)/blue(−) | the visual "where" — look for a coherent patch per zone |

> ⚠️ *Comparable, not identical to clustering:* the gate uses the same σ thresholds but only over
> the window, so windowed qualifier counts are **lower** than whole-ERSP high-activity counts —
> that's expected, and is the point of a zone-restricted confirmatory test.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


In [ ]:
def show(md):
    display(Markdown(md))

runs = P.list_runs()
if not len(runs):
    show('> _no runs yet — execute 410 → 420 → 430 first._')
else:
    display(runs)


## Qualifier summary (from the cached pool table)


In [ ]:
import matplotlib.pyplot as plt
USE_DS = True                       # must match what you ran in 420
GRID   = 'ds' if USE_DS else 'full'
try:
    df_pool = P.load_pool_table(grid=GRID)
    summ = P.qualifier_summary(df_pool)
    display(summ)
    piv = summ.pivot_table(index=['condition', 'zone', 'sign'], columns='window_shape',
                           values='n_contacts', fill_value=0)
    ax = piv.plot.barh(figsize=(9, 0.4 * len(piv) + 1))
    ax.set_xlabel('# qualifying contacts')
    ax.set_title('Qualifiers per zone — boxcar vs gaussian (robustness)')
    plt.tight_layout(); plt.show()
except FileNotFoundError:
    show('> _no pool table yet — run 420._')


## Boxcar vs Gaussian — anatomy robustness
Per-zone purity + compactness side by side for the two window shapes (Yeo-7 latest runs). Close
columns = the anatomical story does not depend on the exact window.


In [ ]:
rows = []
for shape in P.WINDOW_SHAPES:
    rd = P.latest_run('anatomy', 'yeo7', shape)
    if rd is None:
        continue
    anat = pd.read_csv(rd / 'per_cluster_anatomy.csv')
    comp = pd.read_csv(rd / 'per_cluster_spatial_compactness.csv')
    m = anat.merge(comp[['cluster_id', 'distance_mm', 'n_with_coords']], on='cluster_id')
    m['window_shape'] = shape
    rows.append(m)
if rows:
    allm = pd.concat(rows, ignore_index=True)
    cols = ['zone', 'window_shape', 'n_total', 'purity', 'entropy_bits',
            'top_region', 'top_proportion', 'distance_mm']
    cols = [c for c in cols if c in allm.columns]
    display(allm[cols].sort_values(['zone', 'window_shape']).reset_index(drop=True))
else:
    show('> _no anatomy runs yet — run 430._')


## Surface renders (latest)
The most recent fsaverage renders, if `430` produced any.


In [ ]:
ren = P.latest_run('anatomy', 'renders')
if ren is None or not (ren / 'renders').exists():
    show('> _no renders yet — run 430 section 3._')
else:
    for png in sorted((ren / 'renders').rglob('*.png'))[:12]:
        show(f'**{png.relative_to(ren)}**')
        display(Image(filename=str(png)))


# Takeaways & caveats
- **Qualifier counts** answer "does each zone recruit a population of contacts?"; **anatomy
  purity / compactness** answer "do they sit somewhere coherent?".
- **Boxcar vs Gaussian agreement is the robustness signal** — report both; flag any zone where
  they disagree as edge-sensitive.
- **Windowed ≠ whole-ERSP gating.** Lower counts here than clustering's high-activity totals are
  expected — the window is the hypothesis.
- **Sanity check:** `audio` qualifiers should be auditory-network / temporal-gyrus heavy;
  `perception` should be early-window heavy. If not, revisit the windows in `410`.

_To refresh: re-run 420 / 430 (writes new runs), then re-run this notebook — it always reads the
latest run per (stage · target · shape)._


# Methods rationale & key references

**What this analysis is called.** Per contact we pool baseline-normalized **event-related
spectral perturbation (ERSP)** power within *a-priori, hypothesis-driven time windows* — a
**confirmatory time–frequency region-of-interest (ROI) analysis**, the counterpart to the
data-driven **cluster-based permutation test** (Maris & Oostenveld 2007). Thresholding each
contact's windowed power against baseline to label it "responsive" is standard intracranial-EEG
practice; high-gamma (70–150 Hz) is the canonical task-response marker
(see also ERD/ERS, Pfurtscheller & Lopes da Silva 1999).

**Why two window shapes.** Pooling power over a window is a weighted temporal average — i.e.
convolving the signal with a kernel, where the window is a *taper* / *apodization* function.
The **boxcar** (rectangular) window weights the zone equally — the straight hypothesis test —
but its hard edges cause spectral leakage / edge sensitivity. The **Gaussian** taper down-weights
the edges, suppressing edge artefacts and tolerating trial-to-trial latency jitter; close
**boxcar ≈ Gaussian** agreement is the robustness signal (Harris 1978). In machine-learning terms
this is **temporal average pooling** (uniform vs Gaussian pooling kernel); the principled
generalization is **multitaper** estimation (Slepian/DPSS tapers; Thomson 1982).

**References**
- Pfurtscheller G & Lopes da Silva FH (1999). Event-related EEG/MEG synchronization and
  desynchronization: basic principles. *Clin. Neurophysiol.* 110(11):1842–1857.
- Maris E & Oostenveld R (2007). Nonparametric statistical testing of EEG- and MEG-data.
  *J. Neurosci. Methods* 164(1):177–190.  *(the data-driven alternative to a-priori windows)*
- Harris FJ (1978). On the use of windows for harmonic analysis with the discrete Fourier
  transform. *Proc. IEEE* 66(1):51–83.  *(boxcar vs Gaussian vs other tapers)*
- Hamilton LS, Edwards E & Chang EF (2018). A spatial map of onset and sustained responses to
  speech in the human superior temporal gyrus. *Curr. Biol.* 28(12):1860–1871.
  *(per-electrode time-window response typing + anatomical mapping — closest iEEG analogue)*
- Forseth KJ et al. (2018). A lexical semantic hub for heteromodal naming in middle fusiform
  gyrus. *Brain* 141(7):2112–2126.  *(a-priori windows + electrode responsiveness)*

*Related terms:* ERD/ERS · high-gamma / high-frequency broadband (HFB) · windowed band-power
averaging · time–frequency ROI · tapering / apodization · matched filter · temporal pooling.
